In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge — End-to-End Solution
### MAP@3 ranking of top-3 answers (A–E) for knowledge-based MCQs

**Pipeline overview**

| # | Model | Category | Idea |
|---|-------|----------|------|
| 1 | TF-IDF + PyTorch MLP | **Built from scratch** | No pretrained weights anywhere; learns purely from the ~2,000 training rows |
| 2 | DeBERTa-v3 (`AutoModelForMultipleChoice`) | **Pretrained, fine-tuned** | Transfer learning — pretrained language + world knowledge, adapted to this task |
| 3 | XGBoost on engineered similarity features (TF-IDF sim, Sentence-Transformer sim, lexical overlap, etc.) | **Additional model of choice** | Tree-based, feature-driven — a structurally different failure mode from 1 & 2 |
| — | Weighted ensemble of the three | **Final submission** | Combines probability outputs, tuned on a held-out validation split |

**Notebook structure**
1. Setup & data loading
2. EDA (light)
3. MAP@3 metric implementation
4. Train/validation split
5. Model 1 — from-scratch TF-IDF + MLP
6. Model 2 — pretrained DeBERTa-v3 fine-tuned as multiple-choice classifier
7. Model 3 — XGBoost on engineered similarity features
8. Ensembling + local MAP@3 evaluation
9. Final inference on `test.csv` + submission file
10. (Optional, commented out) Zero-shot LLM prompting extension

> Upload `train.csv`, `test.csv`, and `sample_submission.csv` to the Colab working directory (or mount Google Drive) before running.


## 1. Setup

In [2]:
# Run this once per session.
# NOTE: Kaggle's base image already ships compatible, modern versions of numpy, pandas,
# scikit-learn, and torch (with correct GPU/CUDA support) -- do NOT pin/reinstall those,
# it breaks scipy/sklearn (numpy 1.26 vs scipy built for numpy 2.x -> "numpy.strings" error).
# We only install what is actually missing from the Kaggle image.

!pip install --upgrade pip -q

!pip install \
    transformers==4.46.0 \
    datasets==3.1.0 \
    accelerate==1.1.0 \
    sentence-transformers==3.0.1 \
    xgboost==2.1.1 \
    wandb \
    --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.5 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
tpot 1.1.0 requires xgboost>=3.0.0, but you have xgboost 2.1.1 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you have numba-cuda 0.30.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
# Verify installations
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
import xgboost
import sentence_transformers
import datasets
import accelerate

print("✅ All imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"XGBoost: {xgboost.__version__}")
print(f"Sentence-Transformers: {sentence_transformers.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")

# Check GPU availability
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

2026-07-31 13:47:14.439182: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785505634.808859      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785505634.919478      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785505635.783329      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785505635.783371      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785505635.783374      23 computation_placer.cc:177] computation placer alr

✅ All imports successful!
NumPy: 2.4.6
Pandas: 2.3.3
Scikit-learn: 1.6.1
PyTorch: 2.10.0+cu128
Transformers: 4.46.0
XGBoost: 2.1.1
Sentence-Transformers: 3.0.1
Datasets: 3.1.0
Accelerate: 1.1.0

GPU Available: True
GPU Name: Tesla T4


In [4]:
# ================== Weights & Biases: login ==================
# This uses the Kaggle "Secrets" add-on where you already stored your W&B API key.
# IMPORTANT: replace "WANDB_API_KEY" below with the exact LABEL you gave the secret
# when you added it via Add-ons -> Secrets in the Kaggle notebook editor.
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)

WANDB_PROJECT = "smart-mcq-solver"   # all 3 model runs + ensemble will appear under this one project
WANDB_GROUP   = "mcq-ensemble-v1"    # groups them together so they're easy to compare on wandb.ai


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3004169 (23f3004169-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
import os, re, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [6]:
# ---- Paths: adjust if your files live elsewhere (e.g. Google Drive) ----
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge/"   # change to "/content/drive/MyDrive/mcq_challenge" if using Drive

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

OPTIONS = ["A", "B", "C", "D", "E"]
print(train.shape, test.shape, sample_submission.shape)
train.head()

(2000, 8) (500, 7) (500, 2)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## 2. Light EDA

In [7]:
print("Missing values (train):\n", train.isnull().sum())
print("\nAnswer class balance:\n", train["answer"].value_counts())
print("\nPrompt length stats (chars):\n", train["prompt"].str.len().describe())


Missing values (train):
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer class balance:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt length stats (chars):
 count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64


## 3. MAP@3 metric

For each question, if the true label appears at rank *k* (1-indexed) among our top-3 predictions,
the score for that question is `1/k`; if it doesn't appear in the top 3, the score is `0`.
The final metric is the mean over all questions.


In [8]:
print("Missing values (train):\n", train.isnull().sum())
print("\nAnswer class balance:\n", train["answer"].value_counts())
print("\nPrompt length stats (chars):\n", train["prompt"].str.len().describe())


Missing values (train):
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer class balance:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt length stats (chars):
 count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64


## 3. MAP@3 metric

For each question, if the true label appears at rank *k* (1-indexed) among our top-3 predictions,
the score for that question is `1/k`; if it doesn't appear in the top 3, the score is `0`.
The final metric is the mean over all questions.


In [9]:
def map_at_3(y_true, top3_preds):
    """
    y_true      : list/array of true labels, e.g. ['A','C',...]
    top3_preds  : list of lists, each an ordered top-3 prediction e.g. [['A','B','C'], ...]
    """
    scores = []
    for true_label, preds in zip(y_true, top3_preds):
        score = 0.0
        for rank, p in enumerate(preds[:3], start=1):
            if p == true_label:
                score = 1.0 / rank
                break
        scores.append(score)
    return float(np.mean(scores))

def probs_to_top3(prob_matrix, classes=OPTIONS):
    """prob_matrix: (n_samples, 5) array of class probabilities in the order of `classes`."""
    order = np.argsort(-prob_matrix, axis=1)  # descending
    top3 = [[classes[idx] for idx in row[:3]] for row in order]
    return top3


## 4. Train / validation split

We hold out 15% of the training data (stratified by answer label) purely for local MAP@3 evaluation
and ensemble-weight tuning. The final models are re-fit on the *full* training set before predicting on `test.csv`.


In [10]:
train_idx, val_idx = train_test_split(
    train.index, test_size=0.15, random_state=SEED, stratify=train["answer"]
)
tr_df  = train.loc[train_idx].reset_index(drop=True)
val_df = train.loc[val_idx].reset_index(drop=True)
print("train:", tr_df.shape, " val:", val_df.shape)


train: (1700, 8)  val: (300, 8)


## 5. Model 1 — Built From Scratch: TF-IDF + PyTorch MLP

No pretrained weights are used anywhere in this model. We:
1. Fit a TF-IDF vectorizer on `prompt + all options` from the training data only.
2. For each (prompt, option) pair, build a feature vector = `[tfidf(prompt) , tfidf(option) , elementwise-product]`
   (a lightweight, from-scratch analogue of an interaction feature — captures lexical overlap without any pretrained embeddings).
3. Train a small MLP classifier from scratch with cross-entropy loss over the 5 options.


In [11]:
def build_corpus(df):
    return pd.concat([df["prompt"], df["A"], df["B"], df["C"], df["D"], df["E"]]).astype(str).tolist()

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 2), stop_words="english")
tfidf.fit(build_corpus(tr_df))

def make_pair_features(df, vectorizer):
    """
    Returns X of shape (n_samples, 5, 3*D) -- one row of features per option, per question.
    D = tfidf dimensionality after an SVD-free direct approach would be huge, so instead we
    just use similarity + raw scores which is far more memory-friendly for a from-scratch MLP.
    """
    prompt_vecs = vectorizer.transform(df["prompt"].astype(str))
    feats = np.zeros((len(df), 5, 4), dtype=np.float32)  # 4 hand-built, from-scratch features per option
    for i, opt in enumerate(OPTIONS):
        opt_vecs = vectorizer.transform(df[opt].astype(str))
        sim = cosine_similarity(prompt_vecs, opt_vecs).diagonal()          # semantic-lexical overlap
        opt_len = df[opt].astype(str).str.len().values
        prompt_len = df["prompt"].astype(str).str.len().values
        len_ratio = opt_len / (prompt_len + 1)
        word_overlap = df.apply(
            lambda r, o=opt: len(set(str(r["prompt"]).lower().split()) & set(str(r[o]).lower().split())),
            axis=1
        ).values
        feats[:, i, 0] = sim
        feats[:, i, 1] = len_ratio
        feats[:, i, 2] = word_overlap
        feats[:, i, 3] = opt_len
    return feats

X_tr_raw  = make_pair_features(tr_df, tfidf)
X_val_raw = make_pair_features(val_df, tfidf)

# Normalize features (per-column) using train statistics only
mu, sigma = X_tr_raw.reshape(-1, 4).mean(0), X_tr_raw.reshape(-1, 4).std(0) + 1e-6
X_tr  = (X_tr_raw  - mu) / sigma
X_val = (X_val_raw - mu) / sigma

le = LabelEncoder().fit(OPTIONS)
y_tr  = le.transform(tr_df["answer"])
y_val = le.transform(val_df["answer"])

print(X_tr.shape, X_val.shape)


(1700, 5, 4) (300, 5, 4)


In [12]:
class MCQFeatureDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

class ScratchMLP(nn.Module):
    """From-scratch model: a small MLP that scores each of the 5 options."""
    def __init__(self, n_features=4, hidden=64):
        super().__init__()
        self.option_scorer = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        # x: (batch, 5, n_features) -> score per option -> (batch, 5)
        b, n_opts, n_feat = x.shape
        scores = self.option_scorer(x.reshape(-1, n_feat)).reshape(b, n_opts)
        return scores

train_ds = MCQFeatureDataset(X_tr, y_tr)
val_ds   = MCQFeatureDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)

model1 = ScratchMLP().to(DEVICE)
optimizer = torch.optim.Adam(model1.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 40

# ================== W&B: start run for Model 1 ==================
run1 = wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="model1-tfidf-mlp",
    job_type="train",
    config={
        "model": "ScratchMLP (from scratch)",
        "n_features": 4,
        "hidden": 64,
        "epochs": EPOCHS,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "optimizer": "Adam",
    },
    reinit=True,
)
best_val_map3 = -1
best_state = None

for epoch in range(EPOCHS):
    model1.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model1(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    model1.eval()
    with torch.no_grad():
        val_logits = model1(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
        val_probs = F.softmax(val_logits, dim=1).cpu().numpy()
    top3 = probs_to_top3(val_probs)
    val_map3 = map_at_3(val_df["answer"].tolist(), top3)

    # ---- W&B: compute accuracy / F1 and log this epoch ----
    val_pred_labels = [OPTIONS[i] for i in np.argmax(val_probs, axis=1)]
    val_acc = accuracy_score(val_df["answer"].tolist(), val_pred_labels)
    val_f1  = f1_score(val_df["answer"].tolist(), val_pred_labels, average="macro")
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": total_loss / len(train_ds),
        "val_map3": val_map3,
        "val_accuracy": val_acc,
        "val_f1": val_f1,
    })

    if val_map3 > best_val_map3:
        best_val_map3 = val_map3
        best_state = {k: v.clone() for k, v in model1.state_dict().items()}

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d} | train_loss={total_loss/len(train_ds):.4f} | val_MAP@3={val_map3:.4f}")

model1.load_state_dict(best_state)
print("\nModel 1 (from scratch) best val MAP@3:", round(best_val_map3, 4))

# ---- W&B: log final summary metrics and close this run ----
wandb.summary["best_val_map3"] = best_val_map3
wandb.summary["best_val_accuracy"] = val_acc
wandb.summary["best_val_f1"] = val_f1
wandb.finish()


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run p62bny87
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260731_134739-p62bny87
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-tfidf-mlp
wandb: ⭐️ View project at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver/runs/p62bny87


Epoch 01 | train_loss=1.5710 | val_MAP@3=0.5711
Epoch 05 | train_loss=1.4000 | val_MAP@3=0.6289
Epoch 10 | train_loss=1.3676 | val_MAP@3=0.6517
Epoch 15 | train_loss=1.3317 | val_MAP@3=0.6672
Epoch 20 | train_loss=1.3338 | val_MAP@3=0.6639
Epoch 25 | train_loss=1.3111 | val_MAP@3=0.6794
Epoch 30 | train_loss=1.3140 | val_MAP@3=0.6656
Epoch 35 | train_loss=1.3095 | val_MAP@3=0.6706


wandb: uploading wandb-metadata.json; updating run metadata


Epoch 40 | train_loss=1.2882 | val_MAP@3=0.6728

Model 1 (from scratch) best val MAP@3: 0.6806


wandb: uploading config.yaml; uploading output.log
wandb: uploading history steps 0-39, summary, console lines 0-10
wandb: 
wandb: Run history:
wandb:        epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
wandb:   train_loss █▆▅▄▄▄▃▃▄▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁
wandb: val_accuracy ▁▂▃▄▄▅▅▅▆▆▆▆▆▇▇▆▇▇▆▇▇▇████▇█▆▆▇▇▆▇▇▇▇▇▇▇
wandb:       val_f1 ▁▂▃▄▄▅▅▅▅▆▆▆▆▇▇▆▇▇▆▆▇▇████▇█▆▆▇▇▆▆▇▇▇▇▇▇
wandb:     val_map3 ▁▂▄▄▅▆▅▅▆▆▆▇▆▇▇▆▇▇▇▇▇▇██████▇▇▇▇▇▇▇▇████
wandb: 
wandb: Run summary:
wandb: best_val_accuracy 0.53333
wandb:       best_val_f1 0.52583
wandb:     best_val_map3 0.68056
wandb:             epoch 40
wandb:        train_loss 1.28824
wandb:      val_accuracy 0.53333
wandb:            val_f1 0.52583
wandb:          val_map3 0.67278
wandb: 
wandb: 🚀 View run model1-tfidf-mlp at: https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver/runs/p62bny87
wandb: ⭐️ View project at: https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s)

## 6. Model 2 — Pretrained, Fine-Tuned: DeBERTa-v3 Multiple-Choice

We use HuggingFace's `AutoModelForMultipleChoice` with a pretrained checkpoint
(`microsoft/deberta-v3-base` by default — swap to `microsoft/deberta-v3-small` if you have limited GPU memory,
or a larger checkpoint if you have an A100 and want to push MAP@3 higher).

Each training example is turned into 5 `(prompt, option)` pairs; the model outputs one logit per option
and is trained with cross-entropy over the correct index — the standard HF multiple-choice recipe.


In [13]:
from transformers import (
    AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer,
    default_data_collator
)
from datasets import Dataset as HFDataset

MODEL_CKPT = "microsoft/deberta-v3-base"   # try "microsoft/deberta-v3-small" for a faster/lighter run
MAX_LEN = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def to_hf_format(df, has_label=True):
    records = []
    for _, row in df.iterrows():
        rec = {
            "prompt": row["prompt"],
            "opts": [row[o] for o in OPTIONS],
        }
        if has_label:
            rec["label"] = OPTIONS.index(row["answer"])
        records.append(rec)
    return HFDataset.from_list(records)

hf_train = to_hf_format(tr_df, has_label=True)
hf_val   = to_hf_format(val_df, has_label=True)

def preprocess(examples):
    first_sentences = [[p] * 5 for p in examples["prompt"]]
    second_sentences = examples["opts"]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences, second_sentences,
        truncation=True, max_length=MAX_LEN, padding="max_length"
    )
    n = len(examples["prompt"])
    out = {k: [v[i*5:(i+1)*5] for i in range(n)] for k, v in tokenized.items()}
    if "label" in examples:
        out["label"] = examples["label"]
    return out

hf_train_tok = hf_train.map(preprocess, batched=True, remove_columns=hf_train.column_names)
hf_val_tok   = hf_val.map(preprocess, batched=True, remove_columns=hf_val.column_names)


# ================== W&B: metric function for the HF Trainer ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [14]:
# ================== W&B: start run for Model 2 ==================
run2 = wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="model2-deberta-v3",
    job_type="train",
    config={
        "model": MODEL_CKPT,
        "max_len": MAX_LEN,
        "epochs": 4,
        "lr": 2e-5,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "weight_decay": 0.01,
    },
    reinit=True,
)

model2 = AutoModelForMultipleChoice.from_pretrained(MODEL_CKPT).to(DEVICE)

args = TrainingArguments(
    output_dir="./deberta_mcq",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to="wandb",       # <-- sends every training/eval loss straight to the W&B run above
    run_name="model2-deberta-v3",
    seed=SEED,
)

trainer = Trainer(
    model=model2,
    args=args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_val_tok,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,   # <-- makes HF log eval_accuracy / eval_f1 to W&B each epoch
)

trainer.train()


wandb: setting up run 5p8cs2i4
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260731_134755-5p8cs2i4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-deberta-v3
wandb: ⭐️ View project at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver/runs/5p8cs2i4


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
0,5.948000,0.981395,0.813333,0.808232
1,2.321100,0.301211,0.970000,0.970409
2,1.158900,0.161161,0.983333,0.984283
3,0.711100,0.117351,0.983333,0.984283


TrainOutput(global_step=212, training_loss=2.7449747751343927, metrics={'train_runtime': 899.561, 'train_samples_per_second': 7.559, 'train_steps_per_second': 0.236, 'total_flos': 3340882543587840.0, 'train_loss': 2.7449747751343927, 'epoch': 3.981220657276995})

In [15]:
def get_deberta_probs(df, has_label=False):
    hf_ds = to_hf_format(df, has_label=has_label)
    cols_to_remove = hf_ds.column_names
    hf_tok = hf_ds.map(preprocess, batched=True, remove_columns=cols_to_remove)
    preds = trainer.predict(hf_tok)
    logits = preds.predictions
    probs = F.softmax(torch.tensor(logits), dim=1).numpy()
    return probs

val_probs_m2 = get_deberta_probs(val_df, has_label=True)
top3_m2 = probs_to_top3(val_probs_m2)
val_map3_m2 = map_at_3(val_df["answer"].tolist(), top3_m2)
print("Model 2 (pretrained DeBERTa-v3, fine-tuned) val MAP@3:", round(val_map3_m2, 4))

# ---- W&B: log final MAP@3 (accuracy/F1 were already logged every epoch via compute_metrics) and close ----
val_pred_labels_m2 = [OPTIONS[i] for i in np.argmax(val_probs_m2, axis=1)]
val_acc_m2 = accuracy_score(val_df["answer"].tolist(), val_pred_labels_m2)
val_f1_m2  = f1_score(val_df["answer"].tolist(), val_pred_labels_m2, average="macro")
wandb.summary["best_val_map3"] = val_map3_m2
wandb.summary["best_val_accuracy"] = val_acc_m2
wandb.summary["best_val_f1"] = val_f1_m2
wandb.finish()


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

wandb: updating run metadata


Model 2 (pretrained DeBERTa-v3, fine-tuned) val MAP@3: 0.9911


wandb: uploading output.log; uploading config.yaml
wandb: uploading history steps 12-13, summary, console lines 2-2
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▇██
wandb:                 eval/f1 ▁▇██
wandb:               eval/loss █▂▁▁
wandb:            eval/runtime █▅▃▁
wandb: eval/samples_per_second ▁▄▆█
wandb:   eval/steps_per_second ▁▄▆█
wandb:           test/accuracy ▁
wandb:                 test/f1 ▁
wandb:               test/loss ▁
wandb:            test/runtime ▁
wandb:                      +7 ...
wandb: 
wandb: Run summary:
wandb:       best_val_accuracy 0.98333
wandb:             best_val_f1 0.98428
wandb:           best_val_map3 0.99111
wandb:           eval/accuracy 0.98333
wandb:                 eval/f1 0.98428
wandb:               eval/loss 0.11735
wandb:            eval/runtime 11.8423
wandb: eval/samples_per_second 25.333
wandb:   eval/steps_per_second 1.604
wandb:           test/accuracy 0.98333
wandb:                     +15 ...
wandb: 
wandb: 🚀 View r

## 7. Model 3 — Additional Model of Choice: XGBoost on Engineered Similarity Features

This is a completely different mechanism from Models 1 and 2: no gradient descent through raw text at all.
Instead we hand-engineer features per `(prompt, option)` pair using:
- TF-IDF cosine similarity
- Pretrained **Sentence-Transformer** embedding cosine similarity (`all-MiniLM-L6-v2` — used *zero-shot*, not fine-tuned)
- Lexical overlap ratio, length ratio, and the option's **relative rank** among its 4 siblings for each of the above

...then train a multiclass XGBoost classifier (5-way, `multi:softprob`) on these tabular features.


In [16]:
from sentence_transformers import SentenceTransformer
import xgboost as xgb

sbert = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def sbert_embed(texts, batch_size=64):
    return sbert.encode(list(texts), batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True)

def build_xgb_features(df, tfidf_vectorizer):
    prompt_tfidf = tfidf_vectorizer.transform(df["prompt"].astype(str))
    prompt_emb = sbert_embed(df["prompt"].astype(str).tolist())

    tfidf_sims, sbert_sims, len_ratios, word_overlaps = [], [], [], []
    for opt in OPTIONS:
        opt_tfidf = tfidf_vectorizer.transform(df[opt].astype(str))
        tfidf_sims.append(cosine_similarity(prompt_tfidf, opt_tfidf).diagonal())

        opt_emb = sbert_embed(df[opt].astype(str).tolist())
        num = (prompt_emb * opt_emb).sum(axis=1)
        denom = (np.linalg.norm(prompt_emb, axis=1) * np.linalg.norm(opt_emb, axis=1) + 1e-8)
        sbert_sims.append(num / denom)

        opt_len = df[opt].astype(str).str.len().values
        prompt_len = df["prompt"].astype(str).str.len().values
        len_ratios.append(opt_len / (prompt_len + 1))

        overlap = df.apply(
            lambda r, o=opt: len(set(str(r["prompt"]).lower().split()) & set(str(r[o]).lower().split())),
            axis=1
        ).values
        word_overlaps.append(overlap)

    tfidf_sims = np.stack(tfidf_sims, axis=1)   # (n, 5)
    sbert_sims = np.stack(sbert_sims, axis=1)
    len_ratios = np.stack(len_ratios, axis=1)
    word_overlaps = np.stack(word_overlaps, axis=1)

    # relative rank of each option among its 4 siblings (normalized 0-1), adds a 'competition' signal
    def rel_rank(mat):
        order = mat.argsort(axis=1).argsort(axis=1)
        return order / (mat.shape[1] - 1)

    feats_per_option = []
    for i in range(5):
        f = np.stack([
            tfidf_sims[:, i], sbert_sims[:, i], len_ratios[:, i], word_overlaps[:, i],
            rel_rank(tfidf_sims)[:, i], rel_rank(sbert_sims)[:, i],
        ], axis=1)
        feats_per_option.append(f)

    # stacked as (n_samples * 5, n_feats) with a repeated row-id so XGBoost sees one row per option
    n = len(df)
    X = np.concatenate(feats_per_option, axis=0)          # (5n, 6)
    option_id = np.repeat(np.arange(5), n)                 # which option (0=A..4=E) each row is
    X = np.concatenate([X, option_id.reshape(-1, 1)], axis=1)
    row_id = np.tile(np.arange(n), 5)                      # maps back to original question index
    return X, row_id

X_tr_xgb, rowid_tr   = build_xgb_features(tr_df, tfidf)
X_val_xgb, rowid_val = build_xgb_features(val_df, tfidf)

y_tr_xgb = np.concatenate([ (tr_df["answer"].values == OPTIONS[i]).astype(int) for i in range(5) ])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
# ================== W&B: start run for Model 3 ==================
run3 = wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="model3-xgboost",
    job_type="train",
    config={
        "model": "XGBoost (engineered similarity features)",
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    reinit=True,
)

# We train XGBoost as a binary "is this option correct?" classifier over all (question, option) rows,
# then at inference time we softmax-normalize the 5 scores per question back into a probability distribution.
xgb_clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=-1,
)
xgb_clf.fit(X_tr_xgb, y_tr_xgb)

def xgb_probs_for_df(X_xgb, row_id, n_rows):
    raw_scores = xgb_clf.predict_proba(X_xgb)[:, 1]   # P(this option is correct), unnormalized across options
    mat = np.zeros((n_rows, 5))
    for i in range(5):
        mat[:, i] = raw_scores[row_id == i] if False else raw_scores[i * n_rows:(i + 1) * n_rows]
    # softmax-normalize each row so the 5 options sum to 1 (comparable probability scale to models 1 & 2)
    e = np.exp(mat - mat.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

val_probs_m3 = xgb_probs_for_df(X_val_xgb, rowid_val, len(val_df))
top3_m3 = probs_to_top3(val_probs_m3)
val_map3_m3 = map_at_3(val_df["answer"].tolist(), top3_m3)
print("Model 3 (XGBoost, engineered features) val MAP@3:", round(val_map3_m3, 4))

# ---- W&B: compute accuracy / F1, log, and close this run ----
val_pred_labels_m3 = [OPTIONS[i] for i in np.argmax(val_probs_m3, axis=1)]
val_acc_m3 = accuracy_score(val_df["answer"].tolist(), val_pred_labels_m3)
val_f1_m3  = f1_score(val_df["answer"].tolist(), val_pred_labels_m3, average="macro")
wandb.log({"val_map3": val_map3_m3, "val_accuracy": val_acc_m3, "val_f1": val_f1_m3})
wandb.summary["best_val_map3"] = val_map3_m3
wandb.summary["best_val_accuracy"] = val_acc_m3
wandb.summary["best_val_f1"] = val_f1_m3
wandb.finish()


wandb: setting up run be1m07lo
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260731_140327-be1m07lo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3-xgboost
wandb: ⭐️ View project at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver/runs/be1m07lo
wandb: updating run metadata; uploading console lines 0-0


Model 3 (XGBoost, engineered features) val MAP@3: 0.8433


wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 0-0, summary, console lines 0-0
wandb: 
wandb: Run history:
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb:     val_map3 ▁
wandb: 
wandb: Run summary:
wandb: best_val_accuracy 0.74667
wandb:       best_val_f1 0.73322
wandb:     best_val_map3 0.84333
wandb:      val_accuracy 0.74667
wandb:            val_f1 0.73322
wandb:          val_map3 0.84333
wandb: 
wandb: 🚀 View run model3-xgboost at: https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver/runs/be1m07lo
wandb: ⭐️ View project at: https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260731_140327-be1m07lo/logs


## 8. Ensembling

We average the three models' per-class probability distributions (each already softmax-normalized over
the 5 options), sweeping over a small weight grid on the validation set to pick the best combination.


In [18]:
with torch.no_grad():
    val_logits_m1 = model1(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
    val_probs_m1 = F.softmax(val_logits_m1, dim=1).cpu().numpy()

best_weights, best_score = None, -1
for w1 in np.arange(0, 1.01, 0.1):
    for w2 in np.arange(0, 1.01 - w1, 0.1):
        w3 = 1 - w1 - w2
        if w3 < -1e-9:
            continue
        blended = w1 * val_probs_m1 + w2 * val_probs_m2 + w3 * val_probs_m3
        top3 = probs_to_top3(blended)
        score = map_at_3(val_df["answer"].tolist(), top3)
        if score > best_score:
            best_score = score
            best_weights = (w1, w2, w3)

print("Best ensemble weights (model1, model2, model3):", best_weights)
print("Best ensemble val MAP@3:", round(best_score, 4))
print()
print(f"Individual val MAP@3 -> Model1: {map_at_3(val_df['answer'].tolist(), probs_to_top3(val_probs_m1)):.4f} | "
      f"Model2: {val_map3_m2:.4f} | Model3: {val_map3_m3:.4f}")


# ================== W&B: log the ensemble as its own run (bonus, 4th run) ==================
ens_pred_labels = [OPTIONS[i] for i in np.argmax(w1 * val_probs_m1 + w2 * val_probs_m2 + w3 * val_probs_m3, axis=1)]
ens_acc = accuracy_score(val_df["answer"].tolist(), ens_pred_labels)
ens_f1  = f1_score(val_df["answer"].tolist(), ens_pred_labels, average="macro")

run_ens = wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="model4-weighted-ensemble",
    job_type="eval",
    config={"model": "Weighted ensemble (m1+m2+m3)", "w1": best_weights[0], "w2": best_weights[1], "w3": best_weights[2]},
    reinit=True,
)
wandb.log({"val_map3": best_score, "val_accuracy": ens_acc, "val_f1": ens_f1})
wandb.summary["best_val_map3"] = best_score
wandb.summary["best_val_accuracy"] = ens_acc
wandb.summary["best_val_f1"] = ens_f1
wandb.finish()


Best ensemble weights (model1, model2, model3): (np.float64(0.1), np.float64(0.6000000000000001), np.float64(0.29999999999999993))
Best ensemble val MAP@3: 0.9983

Individual val MAP@3 -> Model1: 0.6806 | Model2: 0.9911 | Model3: 0.8433


wandb: setting up run 544ykb32
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260731_140331-544ykb32
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model4-weighted-ensemble
wandb: ⭐️ View project at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/23f3004169-iit-madras/smart-mcq-solver/runs/544ykb32
wandb: updating run metadata; uploading summary
wandb: uploading summary; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb:     val_map3 ▁
wandb: 
wandb: Run summary:
wandb: best_val_accuracy 0.55333
wandb:       best_val_f1 0.54597
wandb:     best_val_map3 0.99833
wandb:      val_accuracy 0.55333
wandb:            val_f1 0.54597
wandb:          val_map3 0.99833
wandb: 
wandb: 🚀 View 

## 9. Final training on full data + inference on `test.csv`

Once you're happy with the validation MAP@3, refit the models on the **full** training set
(train + held-out val combined) so the final submission benefits from all available labeled data,
then predict on `test.csv` using the tuned ensemble weights.

> For brevity this section reuses the already-fit `tfidf`, `model1`, `trainer`/`model2`, and `xgb_clf`
> objects as-is (fit on `tr_df`). For best results, re-run steps 5–7 on the **full** `train` DataFrame
> before this cell, then re-run this inference cell.


In [19]:
from transformers.integrations import WandbCallback
trainer.remove_callback(WandbCallback)

In [20]:
# --- Model 1 predictions on test ---
X_test_raw = make_pair_features(test, tfidf)
X_test = (X_test_raw - mu) / sigma
model1.eval()
with torch.no_grad():
    test_logits_m1 = model1(torch.tensor(X_test, dtype=torch.float32).to(DEVICE))
    test_probs_m1 = F.softmax(test_logits_m1, dim=1).cpu().numpy()

# --- Model 2 predictions on test ---
test_probs_m2 = get_deberta_probs(test, has_label=False)

# --- Model 3 predictions on test ---
X_test_xgb, rowid_test = build_xgb_features(test, tfidf)
test_probs_m3 = xgb_probs_for_df(X_test_xgb, rowid_test, len(test))

# --- Weighted ensemble using the weights tuned in Step 8 ---
w1, w2, w3 = best_weights
test_probs_ensemble = w1 * test_probs_m1 + w2 * test_probs_m2 + w3 * test_probs_m3
test_top3 = probs_to_top3(test_probs_ensemble)

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(p) for p in test_top3]
})
submission.to_csv("submission.csv", index=False)
submission.head()


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

,ID,Prediction
0,1,A B C
1,2,B E D
2,3,B E C
3,4,E B D
4,5,C A D
